In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from torch.utils.data import DataLoader, random_split
from src.transforms import test_transforms
from src.dataset import ImageDataset
from src.configs import BATCH_SIZE, SEED
from pathlib import Path
from src.model import Model
import torch

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# load best checkpoint
best_checkpoint_path = Path("../outputs/checkpoints/checkpoint_epoch_70.pth")
best_checkpoint = torch.load(best_checkpoint_path)

# load parameters into the model
model = Model().to(device, non_blocking=True)
model.load_state_dict(best_checkpoint["model_state_dict"])

# load test dataset and create dataloader
annot_file_test = Path("../data/preprocessed/test/annotations.csv")
img_dir_test = Path("../data/preprocessed/test/Images")

test_dataset = ImageDataset(annot_file_test, img_dir_test,
                            transform=test_transforms)

generator_ = torch.Generator().manual_seed(SEED)
test_dataset, _ = random_split(test_dataset, [0.2, 0.8], generator=generator_)

test_dl = DataLoader(test_dataset, batch_size=BATCH_SIZE, num_workers=4,
                     pin_memory=True, persistent_workers=True)

In [4]:
len(test_dataset)

991

In [5]:
# from src.postprocessing import postprocess_preds

# model.eval()
# with (torch.no_grad()):
#     for X_batch, y_batch in test_dl:
#         X_batch = X_batch.to(device, non_blocking=True)
#         y_batch = y_batch.to(device, non_blocking=True)

#         # 1. forward pass
#         preds_batch = model(X_batch)

#         # 2. process preds
#         for preds in preds_batch:
#             postprocessed_preds = postprocess_preds(preds)

#             print(postprocessed_preds)

{'person': [(tensor(0.6393, device='cuda:0'), 168, 44, 209, 191)], 'pottedplant': [(tensor(0.9865, device='cuda:0'), 74, 94, 148, 204)]}
{}


TypeError: max() received an invalid combination of arguments - got (int, int), but expected one of:
 * (Tensor input, *, Tensor out = None)
 * (Tensor input, Tensor other, *, Tensor out = None)
 * (Tensor input, int dim, bool keepdim = False, *, tuple of Tensors out = None)


In [ ]:
X_batch, y_batch = next(iter(test_dl))

preds = model(X_batch.to(device, non_blocking=True))
postprocessed_preds = postprocess_preds(preds[0], True)

In [ ]:
image = X_batch[0]
image.shape

In [ ]:
from src.transforms import revert_normalization, revert_standardization

revert_standardization(image)
image = revert_normalization(image)

In [ ]:
postprocessed_preds

In [ ]:
coords_list, labels = [], []

for class_name, preds in postprocessed_preds.items():
    for pred in preds:
        x1 = math.ceiling(pred[1].item())
        y1 = mapred[2].item()
        x2 = pred[3].item()
        y2 = pred[4].item()
        coords_list.append((x1, y1, x2, y2))
        labels.append(class_name)

coords_list, labels

In [ ]:
from matplotlib import pyplot as plt
from src.visualization import draw_rectangles

fig, ax = plt.subplots(1, figsize=(8, 8))

image = draw_rectangles(image, 
ax.imshow(image.permute(1, 2, 0))
ax.axis(False)

plt.show()